<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Segmentation</b></h1>
</div>

This notebook executes the classical segmentation experiments covering thresholding, morphology, connected components, contours, color segmentation, distance-transform/watershed separation, quantitative mask evaluation, and validation.


## Setup — Environment and Configuration

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from scipy import ndimage

import cv2

np.set_printoptions(precision=3, suppress=True)

print("NumPy :", np.__version__)
print("OpenCV:", cv2.__version__)
print("Setup : PASS")

### 0.1 Locate the Lab Automatically

Resolve repository-relative data and output paths.

In [ ]:
def find_lab_root():
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "hand.png").exists():
            return candidate

        nested = (
            candidate
            / "Lab_Works"
            / "Image_Processing"
            / "Image_Segmentation"
        )

        if (nested / "data" / "hand.png").exists():
            return nested

    raise FileNotFoundError(
        "Could not locate Image_Segmentation."
    )


LAB_ROOT = find_lab_root()
DATA_DIR = LAB_ROOT / "data"
OUTPUT_DIR = LAB_ROOT / "outputs" / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Lab root:", LAB_ROOT)
print("Data dir:", DATA_DIR)
print("Output  :", OUTPUT_DIR)

### 0.2 Reusable helpers

In [ ]:
def load_gray(path):
    return np.asarray(
        Image.open(path).convert("L"),
        dtype=np.uint8
    )


def load_rgb(path):
    return np.asarray(
        Image.open(path).convert("RGB"),
        dtype=np.uint8
    )


def show_gray(ax, image, title):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")


def show_rgb(ax, image, title):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")


def save_figure(fig, filename):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=160, bbox_inches="tight")
    print("Saved:", path.name)


def to_uint8_mask(mask):
    return (
        np.asarray(mask).astype(bool) * 255
    ).astype(np.uint8)


def overlay_mask(image_rgb, mask, alpha=0.35):
    output = image_rgb.astype(np.float32).copy()
    mask_bool = np.asarray(mask).astype(bool)

    red = np.zeros_like(output)
    red[..., 0] = 255

    output[mask_bool] = (
        (1 - alpha) * output[mask_bool]
        + alpha * red[mask_bool]
    )

    return np.clip(output, 0, 255).astype(np.uint8)

## 1. Segmentation Problem Formulation

## 2. Load the Lab Images

Load and validate the supplied segmentation images.


In [ ]:
hand = load_gray(
    DATA_DIR / "hand.png"
)

tower_rgb = load_rgb(
    DATA_DIR / "tower.jpg"
)

peppers_rgb = load_rgb(
    DATA_DIR / "peppers.png"
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Hand"
)

show_rgb(
    axes[1],
    tower_rgb,
    "Tower"
)

show_rgb(
    axes[2],
    peppers_rgb,
    "Peppers"
)

fig.tight_layout()
save_figure(
    fig,
    "01_input_images.png"
)
plt.show()

print("hand shape   :", hand.shape)
print("tower shape  :", tower_rgb.shape)
print("peppers shape:", peppers_rgb.shape)

## 3. Histogram-Based Threshold Selection

Compute the reference histogram and identify candidate threshold regions.


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4)
)

show_gray(
    axes[0],
    hand,
    "Hand image"
)

axes[1].hist(
    hand.ravel(),
    bins=256,
    range=(0, 255)
)
axes[1].set_title(
    "Hand intensity histogram"
)
axes[1].set_xlabel(
    "Intensity"
)
axes[1].set_ylabel(
    "Pixel count"
)

fig.tight_layout()
save_figure(
    fig,
    "02_hand_histogram.png"
)
plt.show()

## 4. Manual Global Thresholding

Evaluate the selected manual threshold values on a common input.


In [ ]:
manual_threshold = 120

mask_manual = hand < manual_threshold

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    mask_manual,
    f"Mask — T={manual_threshold}"
)

masked_hand = np.where(
    mask_manual,
    hand,
    255
)

show_gray(
    axes[2],
    masked_hand,
    "Segmented foreground"
)

fig.tight_layout()
save_figure(
    fig,
    "03_manual_threshold.png"
)
plt.show()

## 5. Threshold Sensitivity

Run the threshold sweep and record mask-area/component behavior.


In [ ]:
thresholds = [
    70,
    100,
    130,
    160
]

fig, axes = plt.subplots(
    1,
    len(thresholds),
    figsize=(16, 4)
)

for ax, threshold in zip(
    axes,
    thresholds
):
    mask = hand < threshold

    show_gray(
        ax,
        mask,
        f"T={threshold}"
    )

fig.tight_layout()
save_figure(
    fig,
    "04_threshold_sensitivity.png"
)
plt.show()

## 6. Otsu Thresholding

Apply Otsu segmentation and compare it with the manual operating range.


In [ ]:
otsu_threshold, mask_otsu_cv = cv2.threshold(
    hand,
    0,
    255,
    cv2.THRESH_BINARY_INV
    + cv2.THRESH_OTSU
)

mask_otsu = (
    mask_otsu_cv > 0
)

print(
    "Otsu threshold:",
    otsu_threshold
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    mask_otsu,
    f"Otsu mask — T={otsu_threshold:.1f}"
)

overlay = overlay_mask(
    np.stack([hand] * 3, axis=-1),
    mask_otsu
)

show_rgb(
    axes[2],
    overlay,
    "Mask overlay"
)

fig.tight_layout()
save_figure(
    fig,
    "05_otsu_threshold.png"
)
plt.show()

## 7. Gaussian Smoothing Before Thresholding

Evaluate smoothing-before-thresholding under controlled scales.


In [ ]:
hand_blurred = cv2.GaussianBlur(
    hand,
    (5, 5),
    0
)

blur_otsu_threshold, blur_otsu_cv = cv2.threshold(
    hand_blurred,
    0,
    255,
    cv2.THRESH_BINARY_INV
    + cv2.THRESH_OTSU
)

blur_otsu = (
    blur_otsu_cv > 0
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    hand_blurred,
    "Gaussian-smoothed"
)

show_gray(
    axes[2],
    blur_otsu,
    "Otsu after smoothing"
)

fig.tight_layout()
save_figure(
    fig,
    "06_smoothing_before_otsu.png"
)
plt.show()

## 8. Adaptive Thresholding

Compare adaptive mean and Gaussian threshold configurations.


In [ ]:
adaptive_mean = cv2.adaptiveThreshold(
    hand,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY_INV,
    31,
    5
)

adaptive_gaussian = cv2.adaptiveThreshold(
    hand,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    31,
    5
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    adaptive_mean,
    "Adaptive mean"
)

show_gray(
    axes[2],
    adaptive_gaussian,
    "Adaptive Gaussian"
)

fig.tight_layout()
save_figure(
    fig,
    "07_adaptive_thresholding.png"
)
plt.show()

## 9. Morphological Processing

## 10. Structuring Elements

Construct and compare the specified structuring-element shapes and sizes.


In [ ]:
kernel_rect = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (7, 7)
)

kernel_ellipse = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (7, 7)
)

kernel_cross = cv2.getStructuringElement(
    cv2.MORPH_CROSS,
    (7, 7)
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10, 3)
)

show_gray(
    axes[0],
    kernel_rect,
    "Rectangle"
)

show_gray(
    axes[1],
    kernel_ellipse,
    "Ellipse"
)

show_gray(
    axes[2],
    kernel_cross,
    "Cross"
)

fig.tight_layout()
save_figure(
    fig,
    "08_structuring_elements.png"
)
plt.show()

## 11. Erosion and Dilation

Apply erosion and dilation and measure their effect on region support.


In [ ]:
mask_uint8 = to_uint8_mask(
    blur_otsu
)

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (7, 7)
)

eroded = cv2.erode(
    mask_uint8,
    kernel,
    iterations=1
)

dilated = cv2.dilate(
    mask_uint8,
    kernel,
    iterations=1
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    mask_uint8,
    "Original mask"
)

show_gray(
    axes[1],
    eroded,
    "Erosion"
)

show_gray(
    axes[2],
    dilated,
    "Dilation"
)

fig.tight_layout()
save_figure(
    fig,
    "09_erosion_dilation.png"
)
plt.show()

## 12. Opening and Closing

Compare opening and closing on the same initial mask.


In [ ]:
opened = cv2.morphologyEx(
    mask_uint8,
    cv2.MORPH_OPEN,
    kernel
)

closed = cv2.morphologyEx(
    mask_uint8,
    cv2.MORPH_CLOSE,
    kernel
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    mask_uint8,
    "Original mask"
)

show_gray(
    axes[1],
    opened,
    "Opening"
)

show_gray(
    axes[2],
    closed,
    "Closing"
)

fig.tight_layout()
save_figure(
    fig,
    "10_opening_closing.png"
)
plt.show()

## 13. Morphological Gradient

Compute and visualize morphological boundary extraction.


In [ ]:
morph_gradient = cv2.morphologyEx(
    mask_uint8,
    cv2.MORPH_GRADIENT,
    kernel
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    mask_uint8,
    "Mask"
)

show_gray(
    axes[1],
    morph_gradient,
    "Morphological gradient"
)

fig.tight_layout()
save_figure(
    fig,
    "11_morphological_gradient.png"
)
plt.show()

## 14. Hole Filling

Fill enclosed holes and validate exterior-background preservation.


In [ ]:
mask_with_holes = blur_otsu.astype(bool)

filled = ndimage.binary_fill_holes(
    mask_with_holes
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    mask_with_holes,
    "Before filling"
)

show_gray(
    axes[1],
    filled,
    "After filling"
)

show_gray(
    axes[2],
    filled.astype(int)
    - mask_with_holes.astype(int),
    "Pixels added"
)

fig.tight_layout()
save_figure(
    fig,
    "12_hole_filling.png"
)
plt.show()

## 15. Connected Components

Label foreground components and compute their basic properties.


In [ ]:
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    to_uint8_mask(filled),
    connectivity=8
)

print(
    "Number of foreground components:",
    num_labels - 1
)

component_areas = stats[
    1:,
    cv2.CC_STAT_AREA
]

print(
    "Foreground areas:",
    component_areas
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

show_gray(
    axes[0],
    filled,
    "Binary mask"
)

axes[1].imshow(
    labels,
    cmap="nipy_spectral"
)
axes[1].set_title(
    "Connected-component labels"
)
axes[1].axis("off")

fig.tight_layout()
save_figure(
    fig,
    "13_connected_components.png"
)
plt.show()

## 16. Remove Small Components

Filter components by area and verify retention of the principal target.


In [ ]:
minimum_area = 500

clean_components = np.zeros_like(
    labels,
    dtype=bool
)

for label_id in range(
    1,
    num_labels
):
    area = stats[
        label_id,
        cv2.CC_STAT_AREA
    ]

    if area >= minimum_area:
        clean_components |= (
            labels == label_id
        )

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    filled,
    "Before area filtering"
)

show_gray(
    axes[1],
    clean_components,
    f"Area ≥ {minimum_area}"
)

fig.tight_layout()
save_figure(
    fig,
    "14_component_area_filtering.png"
)
plt.show()

## 17. Contours

Extract contours and compare contour geometry with component measurements.


In [ ]:
contours, hierarchy = cv2.findContours(
    to_uint8_mask(
        clean_components
    ),
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

hand_rgb = np.stack(
    [hand] * 3,
    axis=-1
)

contour_view = hand_rgb.copy()

cv2.drawContours(
    contour_view,
    contours,
    -1,
    (255, 0, 0),
    2
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    clean_components,
    "Clean mask"
)

show_rgb(
    axes[1],
    contour_view,
    "Detected contours"
)

fig.tight_layout()
save_figure(
    fig,
    "15_contours.png"
)
plt.show()

print(
    "Number of external contours:",
    len(contours)
)

## 18. Region Properties

Measure the specified geometric descriptors for representative regions.


In [ ]:
region_rows = []

for index, contour in enumerate(
    contours,
    start=1
):
    area = cv2.contourArea(
        contour
    )

    perimeter = cv2.arcLength(
        contour,
        True
    )

    x, y, w, h = cv2.boundingRect(
        contour
    )

    moments = cv2.moments(
        contour
    )

    if moments["m00"] != 0:
        cx = moments["m10"] / moments["m00"]
        cy = moments["m01"] / moments["m00"]
    else:
        cx = np.nan
        cy = np.nan

    circularity = (
        4 * np.pi * area
        / (perimeter ** 2)
        if perimeter > 0
        else np.nan
    )

    aspect_ratio = (
        w / h
        if h > 0
        else np.nan
    )

    region_rows.append(
        {
            "component": index,
            "area": area,
            "perimeter": perimeter,
            "cx": cx,
            "cy": cy,
            "width": w,
            "height": h,
            "aspect_ratio": aspect_ratio,
            "circularity": circularity,
        }
    )

for row in region_rows:
    print(row)

## 19. Color Segmentation

Segment the selected HSV color range and report the retained bounds.


In [ ]:
peppers_hsv = cv2.cvtColor(
    peppers_rgb,
    cv2.COLOR_RGB2HSV
)

hue = peppers_hsv[..., 0]
saturation = peppers_hsv[..., 1]
value = peppers_hsv[..., 2]

fig, axes = plt.subplots(
    1,
    4,
    figsize=(16, 4)
)

show_rgb(
    axes[0],
    peppers_rgb,
    "RGB"
)

show_gray(
    axes[1],
    hue,
    "Hue"
)

show_gray(
    axes[2],
    saturation,
    "Saturation"
)

show_gray(
    axes[3],
    value,
    "Value"
)

fig.tight_layout()
save_figure(
    fig,
    "16_hsv_channels.png"
)
plt.show()

### HSV Range Experiment

Evaluate the two hue intervals required to isolate red in OpenCV HSV space.

In [ ]:
lower_red_1 = np.array(
    [0, 80, 50],
    dtype=np.uint8
)

upper_red_1 = np.array(
    [12, 255, 255],
    dtype=np.uint8
)

lower_red_2 = np.array(
    [165, 80, 50],
    dtype=np.uint8
)

upper_red_2 = np.array(
    [179, 255, 255],
    dtype=np.uint8
)

mask_red_1 = cv2.inRange(
    peppers_hsv,
    lower_red_1,
    upper_red_1
)

mask_red_2 = cv2.inRange(
    peppers_hsv,
    lower_red_2,
    upper_red_2
)

red_mask = (
    (mask_red_1 > 0)
    | (mask_red_2 > 0)
)

red_segment = peppers_rgb.copy()
red_segment[~red_mask] = 0

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_rgb(
    axes[0],
    peppers_rgb,
    "Original peppers"
)

show_gray(
    axes[1],
    red_mask,
    "Red-color mask"
)

show_rgb(
    axes[2],
    red_segment,
    "Segmented red regions"
)

fig.tight_layout()
save_figure(
    fig,
    "17_color_segmentation.png"
)
plt.show()

## 20. Edge-Based Segmentation

Construct and evaluate the edge-driven segmentation branch.


In [ ]:
tower_gray = cv2.cvtColor(
    tower_rgb,
    cv2.COLOR_RGB2GRAY
)

tower_blur = cv2.GaussianBlur(
    tower_gray,
    (5, 5),
    0
)

tower_edges = cv2.Canny(
    tower_blur,
    80,
    160
)

edge_kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (5, 5)
)

tower_edges_closed = cv2.morphologyEx(
    tower_edges,
    cv2.MORPH_CLOSE,
    edge_kernel,
    iterations=2
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_rgb(
    axes[0],
    tower_rgb,
    "Tower"
)

show_gray(
    axes[1],
    tower_edges,
    "Canny edges"
)

show_gray(
    axes[2],
    tower_edges_closed,
    "Closed edge map"
)

fig.tight_layout()
save_figure(
    fig,
    "18_edge_based_segmentation.png"
)
plt.show()

## 21. Distance Transform

Compute the distance map and derive sure-foreground markers.


In [ ]:
distance = cv2.distanceTransform(
    to_uint8_mask(clean_components),
    cv2.DIST_L2,
    5
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    clean_components,
    "Binary mask"
)

im = axes[1].imshow(
    distance,
    cmap="viridis"
)

axes[1].set_title(
    "Distance transform"
)

axes[1].axis("off")

fig.colorbar(
    im,
    ax=axes[1],
    fraction=0.046
)

fig.tight_layout()
save_figure(
    fig,
    "19_distance_transform.png"
)
plt.show()

## 22. Watershed Segmentation

Run marker-based watershed and inspect separation quality.


In [ ]:
watershed_input = peppers_rgb.copy()

peppers_gray = cv2.cvtColor(
    peppers_rgb,
    cv2.COLOR_RGB2GRAY
)

_, peppers_binary = cv2.threshold(
    peppers_gray,
    0,
    255,
    cv2.THRESH_BINARY
    + cv2.THRESH_OTSU
)

ws_kernel = np.ones(
    (3, 3),
    np.uint8
)

opening = cv2.morphologyEx(
    peppers_binary,
    cv2.MORPH_OPEN,
    ws_kernel,
    iterations=2
)

sure_bg = cv2.dilate(
    opening,
    ws_kernel,
    iterations=3
)

dist_transform = cv2.distanceTransform(
    opening,
    cv2.DIST_L2,
    5
)

_, sure_fg = cv2.threshold(
    dist_transform,
    0.5 * dist_transform.max(),
    255,
    0
)

sure_fg = np.uint8(
    sure_fg
)

unknown = cv2.subtract(
    sure_bg,
    sure_fg
)

num_markers, markers = cv2.connectedComponents(
    sure_fg
)

markers = markers + 1
markers[unknown == 255] = 0

markers_ws = cv2.watershed(
    cv2.cvtColor(
        watershed_input,
        cv2.COLOR_RGB2BGR
    ),
    markers.copy()
)

watershed_view = watershed_input.copy()
watershed_view[
    markers_ws == -1
] = [255, 0, 0]

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

show_rgb(
    axes[0, 0],
    peppers_rgb,
    "Input"
)

show_gray(
    axes[0, 1],
    opening,
    "Opening"
)

show_gray(
    axes[0, 2],
    sure_bg,
    "Sure background"
)

show_gray(
    axes[1, 0],
    dist_transform,
    "Distance transform"
)

show_gray(
    axes[1, 1],
    sure_fg,
    "Sure foreground"
)

show_rgb(
    axes[1, 2],
    watershed_view,
    "Watershed boundaries"
)

fig.tight_layout()
save_figure(
    fig,
    "20_watershed.png"
)
plt.show()

## 23. Ground Truth and Segmentation Metrics

Compute the required overlap/classification metrics on the test masks.


In [ ]:
def segmentation_metrics(
    ground_truth,
    prediction
):
    gt = np.asarray(
        ground_truth
    ).astype(bool)

    pred = np.asarray(
        prediction
    ).astype(bool)

    tp = np.logical_and(
        gt,
        pred
    ).sum()

    tn = np.logical_and(
        ~gt,
        ~pred
    ).sum()

    fp = np.logical_and(
        ~gt,
        pred
    ).sum()

    fn = np.logical_and(
        gt,
        ~pred
    ).sum()

    epsilon = 1e-12

    accuracy = (
        (tp + tn)
        / (tp + tn + fp + fn + epsilon)
    )

    precision = (
        tp
        / (tp + fp + epsilon)
    )

    recall = (
        tp
        / (tp + fn + epsilon)
    )

    iou = (
        tp
        / (tp + fp + fn + epsilon)
    )

    dice = (
        2 * tp
        / (2 * tp + fp + fn + epsilon)
    )

    return {
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "IoU": iou,
        "Dice": dice,
    }

### Synthetic Ground-Truth Check

Demonstrate metric computation only; no official ground-truth mask is available for `hand.png`.

In [ ]:
demo_gt = clean_components.copy()

demo_prediction = cv2.erode(
    to_uint8_mask(
        clean_components
    ),
    np.ones(
        (7, 7),
        np.uint8
    ),
    iterations=1
) > 0

demo_metrics = segmentation_metrics(
    demo_gt,
    demo_prediction
)

for key, value in demo_metrics.items():
    if isinstance(value, float):
        print(
            f"{key:10s}: {value:.4f}"
        )
    else:
        print(
            f"{key:10s}: {value}"
        )

## 24. Dice vs IoU Relationship

## 25. Under-Segmentation vs Over-Segmentation

## 26. End-to-End Binary Segmentation Pipeline

Execute the retained complete segmentation pipeline and report its parameters.


In [ ]:
def segment_dark_object(
    image_gray,
    blur_kernel=(5, 5),
    minimum_area=500
):
    blurred = cv2.GaussianBlur(
        image_gray,
        blur_kernel,
        0
    )

    threshold_value, binary = cv2.threshold(
        blurred,
        0,
        255,
        cv2.THRESH_BINARY_INV
        + cv2.THRESH_OTSU
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5)
    )

    cleaned = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        kernel
    )

    cleaned = cv2.morphologyEx(
        cleaned,
        cv2.MORPH_CLOSE,
        kernel
    )

    filled = ndimage.binary_fill_holes(
        cleaned > 0
    )

    num_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            to_uint8_mask(filled),
            connectivity=8
        )
    )

    final_mask = np.zeros_like(
        filled,
        dtype=bool
    )

    for label_id in range(
        1,
        num_labels
    ):
        area = stats[
            label_id,
            cv2.CC_STAT_AREA
        ]

        if area >= minimum_area:
            final_mask |= (
                labels == label_id
            )

    return {
        "threshold": threshold_value,
        "blurred": blurred,
        "binary": binary > 0,
        "cleaned": cleaned > 0,
        "filled": filled,
        "mask": final_mask,
    }

In [ ]:
hand_result = segment_dark_object(
    hand,
    minimum_area=500
)

final_hand_mask = hand_result[
    "mask"
]

final_overlay = overlay_mask(
    np.stack(
        [hand] * 3,
        axis=-1
    ),
    final_hand_mask
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

show_gray(
    axes[0, 0],
    hand,
    "Input"
)

show_gray(
    axes[0, 1],
    hand_result["blurred"],
    "Smoothed"
)

show_gray(
    axes[0, 2],
    hand_result["binary"],
    "Otsu threshold"
)

show_gray(
    axes[1, 0],
    hand_result["cleaned"],
    "Morphological cleanup"
)

show_gray(
    axes[1, 1],
    final_hand_mask,
    "Final mask"
)

show_rgb(
    axes[1, 2],
    final_overlay,
    "Final overlay"
)

fig.tight_layout()
save_figure(
    fig,
    "21_complete_pipeline.png"
)
plt.show()

print(
    "Pipeline Otsu threshold:",
    hand_result["threshold"]
)

## 27. Segmentation Method Selection Criteria

## 28. Integrated Segmentation Workflow

## Final Result Summary

All required segmentation experiments, figures, and validation checks are complete.